# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ismayilysfli/FlyRank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

My lane is Refresh / Content Opportunity Scoring. The decision unit for this project is one pseudonymized content page. The warehouse source is fact_content_daily_performance, whose raw grain is one page per client per day; I will aggregate those daily rows into one row per page. March 1–31, 2026 is my feature and decision window, and April 1–30, 2026 is the future outcome window. I will rank pages by whether their April search impressions fall by more than 20% compared with March, using this as a proxy for pages worth reviewing first. I deliberately exclude April performance from the feature set because it is not knowable at the March decision moment.

In [9]:
%pip -q install duckdb

import duckdb
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError(
        "Add your Hugging Face READ token to Colab Secrets as HF_TOKEN."
    )

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

APRIL = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

print("DuckDB connected.")
print("March and April warehouse partitions are configured.")

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-31"

OUTCOME_START = "2026-04-01"
OUTCOME_END = "2026-04-30"

print("Feature window:", FEATURE_START, "to", FEATURE_END)
print("Outcome window:", OUTCOME_START, "to", OUTCOME_END)

DuckDB connected.
March and April warehouse partitions are configured.
Feature window: 2026-03-01 to 2026-03-31
Outcome window: 2026-04-01 to 2026-04-30


## 2. Fields: feature / label / context / excluded

Features — March only
- march_impressions — total GSC impressions in March.
- march_clicks — total GSC clicks in March.
- march_ctr — March clicks divided by March impressions.
- march_avg_position — average GSC search position during March.
- march_active_days — number of March days where the page received at least one impression.

**Label / proxy**
- is_declining — 1 when April impressions are more than 20% lower than March impressions, otherwise 0. This is a future observed performance proxy, not proof that the page needs a refresh.

**Context**
- client_hash_id and content_hash_id — pseudonymized identifiers used for grouping and splitting only, never as model features.
- gsc_data_available — used to keep only rows where Search Console data is actually available.

**Excluded**
- April performance metrics and any value calculated from them are excluded from the honest feature set because they occur after the decision moment. I will add one such value temporarily during the leakage demonstration, show why it produces an unrealistically strong score, and then remove it.

In [10]:
FEATURE_COLUMNS = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_active_days",
]

CONTEXT_COLUMNS = [
    "client_hash_id",
    "content_hash_id",
]

LABEL_COLUMN = "is_declining"

print("Features:", FEATURE_COLUMNS)
print("Context:", CONTEXT_COLUMNS)
print("Label:", LABEL_COLUMN)

Features: ['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_active_days']
Context: ['client_hash_id', 'content_hash_id']
Label: is_declining


## 3. Verify it with queries (grain, counts, missing values, windows)
Five-feature frame
I aggregate March 2026 into one row per page and use only information available by the end of March.
- march_impressions — knowable at the decision moment because it uses only March GSC impressions.
- march_clicks — knowable at the decision moment because it uses only March GSC clicks.
- march_ctr — knowable at the decision moment because it is calculated only from March clicks and impressions.
- march_avg_position — knowable at the decision moment because it uses only March search-position observations.
- march_active_days — knowable at the decision moment because it counts only March days with at least one impression.
April performance is used only to create the future outcome label and is not included in the honest feature set.

In [11]:
feature_frame = con.sql(f"""
    WITH march_features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN 1.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
                ELSE NULL
            END AS march_ctr,
            AVG(CASE
                WHEN gsc_avg_position > 0 THEN gsc_avg_position
            END) AS march_avg_position,
            COUNT(DISTINCT CASE
                WHEN gsc_impressions > 0 THEN report_date
            END) AS march_active_days
        FROM {MARCH}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    april_outcome AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM {APRIL}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        m.*,
        a.april_impressions,
        CASE
            WHEN a.april_impressions < 0.8 * m.march_impressions THEN 1
            ELSE 0
        END AS is_declining
    FROM march_features m
    INNER JOIN april_outcome a
        USING (client_hash_id, content_hash_id)
""").df()

print("Feature-frame rows:", len(feature_frame))
print("Decline rate:", round(feature_frame["is_declining"].mean(), 3))

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 100893
Decline rate: 0.515


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,march_active_days,april_impressions,is_declining
0,client_62f4a7e64f5e0096,content_35d979572550dd7f,1423.0,1.0,0.000703,16.669625,31,1180.0,0
1,client_62f4a7e64f5e0096,content_d95e1739253d6a65,208.0,0.0,0.000000,35.865605,27,168.0,0
2,client_62f4a7e64f5e0096,content_405cc2e0475690e9,607.0,0.0,0.000000,30.986972,31,249.0,1
3,client_62f4a7e64f5e0096,content_fdb0d09a710f671c,267.0,0.0,0.000000,24.274523,27,336.0,0
4,client_62f4a7e64f5e0096,content_a7b1f7921f001dce,1489.0,0.0,0.000000,47.826506,31,832.0,1


Deliberate leakage test
I first score the model using only the five March features. Then I deliberately add future_change_pct, which uses April impressions and therefore contains information from the outcome period. Because the decline label is defined from the March-to-April change, this feature leaks the answer. I expect the score to become unrealistically high. After demonstrating the effect, I remove the leaked feature and keep the honest score.

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

features = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_active_days",
]

clean = feature_frame.dropna(subset=features).copy()

train, test = train_test_split(
    clean,
    test_size=0.25,
    random_state=42,
    stratify=clean["is_declining"],
)

# Honest score
honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
honest_model.fit(train[features], train["is_declining"])
honest_score = honest_model.score(test[features], test["is_declining"])

# Deliberately leaks the answer
clean["future_change_pct"] = (
    clean["april_impressions"] - clean["march_impressions"]
) / clean["march_impressions"]

train, test = train_test_split(
    clean,
    test_size=0.25,
    random_state=42,
    stratify=clean["is_declining"],
)

leaked_features = features + ["future_change_pct"]

leaked_model = DecisionTreeClassifier(max_depth=3, random_state=42)
leaked_model.fit(train[leaked_features], train["is_declining"])
leaked_score = leaked_model.score(test[leaked_features], test["is_declining"])

print(f"Honest score: {honest_score:.3f}")
print(f"Leaked score: {leaked_score:.3f}")

# Deletes the leaked column
clean.drop(columns="future_change_pct", inplace=True)

print("Leak removed:", "future_change_pct" not in clean.columns)

Honest score: 0.610
Leaked score: 1.000
Leak removed: True


Adding future_change_pct makes the score unrealistically high because the column uses April performance and therefore directly contains information used to define the label. I remove it and keep only the March features.

In [13]:
# Verification query 1 — grain

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

print("Duplicate rows at report_date × client × content grain:",
      len(grain_check))

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows at report_date × client × content grain: 0


,report_date,client_hash_id,content_hash_id,row_count


In [14]:
# Verification query 2 — slice size and date span

slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS distinct_pages,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {MARCH}
""").df()

slice_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,distinct_pages,min_date,max_date
0,9841378,331437,2026-03-01,2026-03-31


In [15]:
# Verification query 3 — GSC availability

availability_check = con.sql(f"""
    WITH all_rows AS (
        SELECT COUNT(*) AS total_rows
        FROM {MARCH}
    ),
    available_rows AS (
        SELECT
            COUNT(*) AS rows_with_gsc,
            COUNT(DISTINCT content_hash_id) AS pages_with_gsc
        FROM {MARCH}
        WHERE gsc_data_available IS TRUE
    )
    SELECT
        total_rows,
        rows_with_gsc,
        pages_with_gsc,
        ROUND(100.0 * rows_with_gsc / total_rows, 2) AS pct_rows_surviving
    FROM all_rows, available_rows
""").df()

availability_check

,total_rows,rows_with_gsc,pages_with_gsc,pct_rows_surviving
0,9841378,3611061,176738,36.69


## 4. Data limits

Limitation — incomplete and uneven GSC coverage: Only 36.69% of the March page-day rows have gsc_data_available IS TRUE. This means my analysis represents only pages and days with usable Google Search Console data, not the entire content inventory. Missing GSC data must not be interpreted as zero impressions or poor performance. Client history also starts at different times, so the available March–April evidence is not equally complete for every client.

In [16]:
gsc_coverage = availability_check.loc[0, "pct_rows_surviving"]

print(f"March rows with usable GSC data: {gsc_coverage:.2f}%")
print("Missing GSC data is treated as unavailable data, not zero performance.")

March rows with usable GSC data: 36.69%
Missing GSC data is treated as unavailable data, not zero performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.